In [252]:
import sys
sys.path.append("/Users/sujaladhikari/Sujal's Personal/Projects/FedIDS")

In [253]:
import os 
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torch 
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from Model.model import MLP
from torch.optim import Adam
import utils
from sklearn.metrics import classification_report

In [254]:
dataset = pd.read_csv('../silos_datasets/combined_binary_silos.csv')
dataset = dataset.drop(columns = 'Unnamed: 0')
dataset.head(5)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_Binary
0,3.503933,-0.537133,-0.010411,-0.008352,-0.083844,-0.006664,-0.296352,-0.225312,-0.296745,-0.265739,...,0.002768,-0.138676,-0.089986,-0.148282,-0.118883,-0.473122,-0.139396,-0.481016,-0.456046,0
1,3.082472,-0.537134,-0.010411,-0.008352,-0.083844,-0.006664,-0.296352,-0.225312,-0.296745,-0.265739,...,0.002768,-0.138676,-0.089986,-0.148282,-0.118883,-0.473122,-0.139396,-0.481016,-0.456046,0
2,-0.381375,1.660075,-0.000851,-0.002327,-0.011088,-0.000485,0.297979,-0.225312,0.030215,0.309483,...,0.002759,-0.138597,-0.089986,-0.148228,-0.118796,2.082227,-0.139396,2.008201,2.116490,1
3,-0.381375,1.708903,-0.002444,-0.002327,-0.009649,-0.000485,0.320181,-0.225312,0.092251,0.380546,...,0.002768,-0.137106,-0.089986,-0.147205,-0.117154,2.137577,-0.139396,2.062119,2.172213,1
4,-0.381375,1.663726,0.000742,-0.003532,-0.014582,-0.000485,0.279192,-0.225312,-0.024394,0.256750,...,0.002768,-0.137110,-0.089986,-0.147208,-0.117159,2.085302,-0.139396,2.011197,2.119586,1


In [255]:
print(dataset['Label_Binary'].value_counts())
print(f"The dataset has following number {dataset.shape[0]} number of rows and {dataset.shape[1]} number of columns")

Label_Binary
0    556488
1    556488
Name: count, dtype: int64
The dataset has following number 1112976 number of rows and 79 number of columns


In [256]:
random_seed = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [257]:
X = dataset.drop(columns = 'Label_Binary')
X = X.to_numpy()
y = dataset['Label_Binary']
y = y.to_numpy()


In [258]:
## Spliting the data into train, test, and validation splits
X_train , X_vals, y_train, y_vals = train_test_split(X,y , test_size=0.3, random_state=random_seed)
X_val, X_test, y_val, y_test = train_test_split(X_vals, y_vals, test_size=0.5, random_state=random_seed)

In [259]:
### Standard Scaling for all the datas

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_train = torch.tensor(X_train, dtype = torch.float32)
y_train = torch.tensor(y_train, dtype = torch.long)

X_val = scaler.transform(X_val)
X_val = torch.tensor(X_val, dtype = torch.float32)
y_val = torch.tensor(y_val, dtype = torch.long)

X_test = scaler.transform(X_test)
X_test = torch.tensor(X_test, dtype = torch.float32)
y_test = torch.tensor(y_test, dtype = torch.long)



In [260]:
training_data = TensorDataset(X_train,y_train)
testing_data = TensorDataset(X_test, y_test)
validation_data = TensorDataset(X_val, y_val)


In [261]:
training_batch = DataLoader(training_data, batch_size = 64, shuffle = True)
testing_batch = DataLoader(testing_data, batch_size=64, shuffle=False)
validation_batch = DataLoader(validation_data,batch_size = 64, shuffle = False)

In [262]:
print(MLP)

<class 'Model.model.MLP'>


In [263]:
input_size = 78
hidden_layer = [256, 128,64,8]
num_classes = 2

In [264]:
## We will be using the same model that is being used in fedmodel.py for the federated purpose
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(input_size, hidden_layer, num_classes).to(device)
### Optimizer and loss function 
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
loss = nn.CrossEntropyLoss()
num_epoch = 10
model = model.to(device)

In [265]:
### Training, testing and evaluation of the centralized model 
## setting up the training environment:
def train(model, train_loader,validation_loader, criterion, device):
    model.train() ## Starting to train the model 
    training_loss = 0.0 ## Calculating the training loss in terms of floating value
    train_correct = 0 ## Count of total number of correct samples in total 
    training_total = 0 ## Count of the total number of samples checked
    for batch in train_loader: ## Loading the data loader
        samples, classes = batch 
        samples = samples.to(device) ## Converting the sample and classes to the device ('cpu')
        classes = classes.to(device)
        prediciton = model(samples) ## Prediction for the samples in the batch 
        optimizer.zero_grad() ## Started the calculation of the loss 
        loss = criterion(prediciton, classes)
        loss.backward() ## Backward propagation 
        optimizer.step() ## Forward optimizatio 
        training_loss +=loss.item() * samples.size(0) ## Calculating the loss for each batch 
        _,predicted = prediciton.max(1) ## Max prediction for each sample in the batch 
        training_total += classes.size(0) ### Each total in training adds up 
        train_correct += predicted.eq(classes).sum().item() ## Converting the total predicted that are equal to the classes and sum them 
    training_loss = training_loss/len(train_loader.dataset)
    training_accuracy = 100 * train_correct/ training_total 
    val_loss = 0
    val_correct = 0
    val_total = 0 
    model.eval()
    with torch.no_grad(): ## Evaluation started
        for val_batch in validation_loader:
            features, labels = val_batch 
            features = features.to(device) ## Converted the path to device 'cpu'
            labels = labels.to(device)
            val_outputs = model(features) ## Outputs
            val_loss_batch = criterion(val_outputs, labels) ## loss calculated
            val_loss += val_loss_batch.item() * features.size(0) ## for each batch 
            _, val_predicted = val_outputs.max(1) ## Maximum value calcuated
            val_correct += val_predicted.eq(labels).sum().item()
            val_total += labels.size(0) 
    avg_validation_loss = val_loss / len(validation_loader.dataset) ## Average across the sum 
    validation_accuracy = 100 * val_correct/ val_total ## Accuracy 
    return training_loss, training_accuracy , avg_validation_loss, validation_accuracy, val_correct, train_correct

In [266]:
### Creating the evalution function 
def evaluate(model, test_loader, critertion, device):
    model.eval()
    test_loss = 0.0 ## This is for the test loss 
    correct = 0 
    total = 0 
    predictions = []
    true_labels = []
    with torch.no_grad():
        for batch in test_loader:
            samples, labels = batch 
            samples = samples.to(device)
            labels = labels.to(device)
            outputs = model(samples)
            loss = critertion(outputs, labels)
            _, predicted = outputs.max(1)
            test_loss += loss.item() * samples.size(0)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            predictions.extend(predicted.tolist())
            true_labels.extend(labels.tolist())
        
    test_loss = test_loss / len(test_loader.dataset)
    accuracy = 100 * correct / total 

    return test_loss, accuracy, predictions, true_labels

### Since the training process has been done, we will evaluate the model's perfomance 

In [267]:
### On the given epoch we will be training and testing the epcoh 
num_epoch
for epoch in range(num_epoch):
    training_loss, training_accuracy , avg_validation_loss, validation_accuracy, val_correct, train_correct = train(model, training_batch,validation_batch, loss, device)
    test_loss, test_accuracy, predictions, true_labels = evaluate(model, testing_batch, loss, device)
    print(f"Epoch [{epoch+1}/{num_epoch}], Training Loss: {training_loss:4f}, Testing Loss: {test_loss:4f}")
    report = classification_report(true_labels, predictions)
    print(report)
    

Epoch [1/10], Training Loss: 0.074636, Testing Loss: 0.066701
              precision    recall  f1-score   support

           0       1.00      0.95      0.97     83612
           1       0.95      1.00      0.97     83335

    accuracy                           0.97    166947
   macro avg       0.97      0.97      0.97    166947
weighted avg       0.97      0.97      0.97    166947

Epoch [2/10], Training Loss: 0.055236, Testing Loss: 0.050125
              precision    recall  f1-score   support

           0       0.99      0.97      0.98     83612
           1       0.97      0.99      0.98     83335

    accuracy                           0.98    166947
   macro avg       0.98      0.98      0.98    166947
weighted avg       0.98      0.98      0.98    166947

Epoch [3/10], Training Loss: 0.050265, Testing Loss: 0.048275
              precision    recall  f1-score   support

           0       0.99      0.97      0.98     83612
           1       0.97      0.99      0.98     833

### Lets evaluate how does the model do in each silos

In [268]:
def silos_evaluate(model, batch, criterion, device):
    model.eval()
    test_loss = 0.0
    total = 0 
    corrected = 0
    predictions = []
    true_labels = []
    for samples, features in batch:
        samples = samples.to(device)
        features = features.to(device)
        outputs = model(samples)
        loss = criterion(outputs, features)
        _, predicted = outputs.max(1)
        total += features.size(0)
        test_loss += loss.item() * samples.size(0)
        corrected += predicted.eq(features).sum().item()
        predictions.extend(predicted.tolist())
        true_labels.extend(features.tolist())
    
    test_loss = test_loss / len(batch.dataset)
    accuracy = 100 * corrected/total

    return test_loss, accuracy, predictions, true_labels


In [283]:
def batch_maker(dataset):
    dataset = dataset.drop(columns = 'Unnamed: 0', errors='ignore')
    X = dataset.drop(columns = 'Label_Binary')
    X = X.to_numpy()
    y = dataset['Label_Binary']
    y = y.to_numpy()
    X_train , X_test, y_train, y_test = train_test_split(X,y , test_size=0.3, random_state=random_seed)
    X_train = scaler.transform(X_train)
    X_train = torch.tensor(X_train, dtype = torch.float32)
    y_train = torch.tensor(y_train, dtype = torch.long)

    X_test = scaler.transform(X_test)
    X_test = torch.tensor(X_test, dtype = torch.float32)
    y_test = torch.tensor(y_test, dtype = torch.long)

    training_batch = DataLoader(TensorDataset(X_train,y_train), batch_size = 64, shuffle = True)
    testing_batch = DataLoader(TensorDataset(X_test,y_test), batch_size=64, shuffle=False)
    
    return training_batch,testing_batch 

In [285]:
## Loading all the test data in the silos 
siloOne_train, siloOne_test = batch_maker(pd.read_csv('../silos_datasets/siloBinaryOne.csv'))
siloTwo_train, siloTwo_test = batch_maker(pd.read_csv('../silos_datasets/siloBinaryTwo.csv'))
siloThree_train, siloThree_test= batch_maker(pd.read_csv('../silos_datasets/siloBinaryThree.csv'))
siloFour_train, siloFour_test = batch_maker(pd.read_csv('../silos_datasets/siloBinaryFour.csv'))

In [286]:
test_loss, test_accuracy, predictions, true_labels = silos_evaluate(model, siloOne_test, loss, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(true_labels, predictions)
print(report)

 Test Loss: 4.6074, Test Accuracy: 84.1566%
              precision    recall  f1-score   support

           0       0.96      0.71      0.82     75551
           1       0.77      0.97      0.86     75477

    accuracy                           0.84    151028
   macro avg       0.87      0.84      0.84    151028
weighted avg       0.87      0.84      0.84    151028



In [291]:
test_loss, test_accuracy, predictions, true_labels = silos_evaluate(model, siloTwo_test, loss, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(true_labels, predictions)
print(report)

 Test Loss: 5.5748, Test Accuracy: 85.5977%
              precision    recall  f1-score   support

           0       1.00      0.72      0.83     86566
           1       0.78      1.00      0.87     86705

    accuracy                           0.86    173271
   macro avg       0.89      0.86      0.85    173271
weighted avg       0.89      0.86      0.85    173271



In [288]:
test_loss, test_accuracy, predictions, true_labels = silos_evaluate(model, siloThree_test, loss, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(true_labels, predictions)
print(report)

 Test Loss: 2.8710, Test Accuracy: 70.8434%
              precision    recall  f1-score   support

           0       0.71      0.71      0.71      4207
           1       0.70      0.71      0.71      4093

    accuracy                           0.71      8300
   macro avg       0.71      0.71      0.71      8300
weighted avg       0.71      0.71      0.71      8300



In [289]:
test_loss, test_accuracy, predictions, true_labels = silos_evaluate(model, siloFour_test, loss, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(true_labels, predictions)
print(report)

 Test Loss: 2.8445, Test Accuracy: 85.1852%
              precision    recall  f1-score   support

           0       0.99      0.72      0.83       665
           1       0.77      0.99      0.87       631

    accuracy                           0.85      1296
   macro avg       0.88      0.86      0.85      1296
weighted avg       0.88      0.85      0.85      1296



---
### Analyzing the two outperforming silos dataset on the centralized model 


In [275]:
siloThree = pd.read_csv("../silos_datasets/SiloBinaryThree.csv")
siloThree.head(5)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_Binary
0,21,221,2,1,14,0,14,0,7.000000,9.899495,...,32,0.0,0.000,0,0,0.0,0.00000,0,0,1
1,53,612792,1,1,48,148,48,48,48.000000,0.000000,...,20,0.0,0.000,0,0,0.0,0.00000,0,0,0
2,443,110219468,24,27,15380,2414,3050,0,640.833333,1051.835070,...,20,1057008.5,2151750.002,5228249,23580,9962586.9,48857.08944,10000000,9850722,0
3,53,37750,2,2,54,86,27,27,27.000000,0.000000,...,32,0.0,0.000,0,0,0.0,0.00000,0,0,0
4,88,755,2,2,504,2884,252,252,252.000000,0.000000,...,20,0.0,0.000,0,0,0.0,0.00000,0,0,0


In [276]:
print(f"Total number of samples in the silo three data is {siloThree.shape[0]} and total features are {siloThree.shape[1]}")

Total number of samples in the silo three data is 27664 and total features are 79


In [277]:
print(f"The Class distribution of the silo three is:")
print(siloThree['Label_Binary'].value_counts())

The Class distribution of the silo three is:
Label_Binary
1    13832
0    13832
Name: count, dtype: int64


---
### Analyzing the siloFour

In [278]:
siloFour = pd.read_csv("../silos_datasets/SiloBinaryFour.csv")
siloFour.head(5)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_Binary
0,80,117286545,21,18,710,4414,357,0,33.809524,106.783247,...,32,190673.1818,529211.8730,1786307,31007,1.000000e+07,4365.02479,10000000,10000000,0
1,80,5243646,3,1,0,0,0,0,0.000000,0.000000,...,32,0.0000,0.0000,0,0,0.000000e+00,0.00000,0,0,1
2,53,31216,1,1,64,130,64,64,64.000000,0.000000,...,20,0.0000,0.0000,0,0,0.000000e+00,0.00000,0,0,0
3,53,31368,2,2,72,236,36,36,36.000000,0.000000,...,20,0.0000,0.0000,0,0,0.000000e+00,0.00000,0,0,0
4,443,61143903,15,14,982,7291,698,0,65.466667,184.588681,...,32,260864.8333,175721.8448,619555,188857,9.929724e+06,316284.11450,10200000,9298227,0


In [279]:
print(f"Total number of samples in the silo four data is {siloFour.shape[0]} and total features are {siloFour.shape[1]}")

Total number of samples in the silo four data is 4318 and total features are 79


In [280]:
print(f"The Class distribution of the silo four is:")
print(siloFour['Label_Binary'].value_counts())

The Class distribution of the silo four is:
Label_Binary
0    2159
1    2159
Name: count, dtype: int64


In [281]:
## Turning the model into pickel file 
path = '../models/'
log_path = os.path.join(path, 'centralized_model.pickle')
utils.savein_pickle(log_path, model)